# ⚖️ Model Fairness & Bias: Intermediate Analysis

---

**Prerequisites:** Basic Python, familiarity with classification models, know what precision/recall means.

**What you'll learn:**
1. Formal fairness definitions — and why they *conflict*
2. Measuring bias with proper statistical tests (not just eyeballing charts)
3. Intersectional bias (gender × race, not just one dimension)
4. The WinoBias benchmark for coreference bias
5. Calibration fairness — does a 70% confidence actually mean 70%?
6. Writing a reusable `FairnessAuditor` class

---

## 🧭 The Impossibility Theorem — Before We Write Any Code

Here's the uncomfortable truth: **you cannot satisfy all fairness definitions simultaneously** (except in trivial cases). This was proven mathematically by Chouldechova (2017) and Kleinberg et al. (2016).

The three most common definitions:

| Fairness Criterion | Definition | Intuition |
|---|---|---|
| **Demographic Parity** | P(Ŷ=1 \| A=0) = P(Ŷ=1 \| A=1) | Same positive rate across groups |
| **Equalized Odds** | P(Ŷ=1 \| Y=1, A=a) equal ∀a | Same TPR *and* FPR across groups |
| **Calibration** | P(Y=1 \| score=s, A=a) = s | Confidence scores mean the same thing for all groups |

You'll see these tensions play out in the experiments below.

In [ ]:
!pip install datasets transformers torch pandas matplotlib seaborn scikit-learn scipy fairlearn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)
from sklearn.calibration import calibration_curve
from datasets import load_dataset
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2E86AB', '#E84855', '#3BB273', '#F6AE2D', '#8B2FC9']
sns.set_palette(PALETTE)

print("✅ Ready.")

---
## 🏗️ Part 0: Build a Reusable `FairnessAuditor`

Before running experiments, let's build a class that encapsulates all the fairness metrics we'll need. Good engineering practice — and reusable for your own projects.

In [ ]:
class FairnessAuditor:
    """
    A reusable auditor for measuring fairness metrics across demographic groups.
    
    Parameters
    ----------
    df         : DataFrame with columns [y_true, y_pred, y_score, group]
    group_col  : column name identifying the sensitive attribute
    label_col  : column name with ground truth labels (0/1)
    pred_col   : column name with model predictions (0/1)
    score_col  : column name with model confidence scores (0.0–1.0)
    """

    def __init__(self, df, group_col, label_col, pred_col, score_col):
        self.df = df.copy()
        self.group_col = group_col
        self.label_col = label_col
        self.pred_col = pred_col
        self.score_col = score_col
        self.groups = sorted(df[group_col].unique())

    def _group(self, g):
        return self.df[self.df[self.group_col] == g]

    def per_group_metrics(self):
        """Compute standard classification metrics per group."""
        rows = []
        for g in self.groups:
            sub = self._group(g)
            yt, yp, ys = sub[self.label_col], sub[self.pred_col], sub[self.score_col]
            cm = confusion_matrix(yt, yp, labels=[0, 1])
            tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
            rows.append({
                'Group': g, 'N': len(sub),
                'Accuracy': accuracy_score(yt, yp),
                'Precision': precision_score(yt, yp, zero_division=0),
                'Recall (TPR)': recall_score(yt, yp, zero_division=0),
                'F1': f1_score(yt, yp, zero_division=0),
                'FPR': fp / (fp + tn) if (fp + tn) > 0 else 0,
                'AUC': roc_auc_score(yt, ys) if len(yt.unique()) > 1 else float('nan'),
                'Pos Rate': yp.mean(),
            })
        return pd.DataFrame(rows).set_index('Group').round(4)

    def demographic_parity_diff(self):
        """Max difference in positive prediction rates across groups."""
        rates = {g: self._group(g)[self.pred_col].mean() for g in self.groups}
        vals = list(rates.values())
        return max(vals) - min(vals), rates

    def equalized_odds_diff(self):
        """Max difference in TPR and FPR across groups."""
        tprs, fprs = {}, {}
        for g in self.groups:
            sub = self._group(g)
            yt, yp = sub[self.label_col], sub[self.pred_col]
            cm = confusion_matrix(yt, yp, labels=[0, 1])
            if cm.size == 4:
                tn, fp, fn, tp = cm.ravel()
                tprs[g] = tp / (tp + fn) if (tp + fn) > 0 else 0
                fprs[g] = fp / (fp + tn) if (fp + tn) > 0 else 0
        tpr_diff = max(tprs.values()) - min(tprs.values())
        fpr_diff = max(fprs.values()) - min(fprs.values())
        return tpr_diff, fpr_diff, tprs, fprs

    def chi_square_test(self):
        """Test if positive prediction rate differs significantly across groups."""
        contingency = pd.crosstab(self.df[self.group_col], self.df[self.pred_col])
        chi2, p, dof, _ = stats.chi2_contingency(contingency)
        return chi2, p, dof

    def plot_roc_curves(self, ax=None):
        """Plot per-group ROC curves."""
        if ax is None:
            fig, ax = plt.subplots(figsize=(7, 6))
        for g, color in zip(self.groups, PALETTE):
            sub = self._group(g)
            yt, ys = sub[self.label_col], sub[self.score_col]
            if len(yt.unique()) > 1:
                fpr, tpr, _ = roc_curve(yt, ys)
                auc = roc_auc_score(yt, ys)
                ax.plot(fpr, tpr, label=f'{g} (AUC={auc:.2f})', color=color, lw=2)
        ax.plot([0,1],[0,1],'--', color='gray', alpha=0.5)
        ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
        ax.set_title('ROC Curves by Group'); ax.legend()
        return ax

    def summary_report(self):
        dp_diff, dp_rates = self.demographic_parity_diff()
        tpr_diff, fpr_diff, tprs, fprs = self.equalized_odds_diff()
        chi2, p, dof = self.chi_square_test()
        print("=" * 55)
        print("  FAIRNESS AUDIT SUMMARY")
        print("=" * 55)
        print(f"\n📐 Demographic Parity Gap:   {dp_diff:.4f}")
        for g, r in dp_rates.items(): print(f"     {g:<20}: {r:.4f} positive rate")
        print(f"\n⚖️  Equalized Odds:")
        print(f"     TPR gap: {tpr_diff:.4f}")
        print(f"     FPR gap: {fpr_diff:.4f}")
        print(f"\n🔬 Chi-Square Test (is the gap statistically significant?):")
        print(f"     χ²={chi2:.2f}, p={p:.4f}, dof={dof}")
        sig = "YES ⚠️" if p < 0.05 else "NO ✅"
        print(f"     Significant at p<0.05: {sig}")
        print("=" * 55)

print("✅ FairnessAuditor class defined.")

---
## 🧪 Part 1: WinoBias — Coreference Resolution Bias

**WinoBias** is a benchmark that tests whether coreference models associate occupations with the "wrong" gender. For example:

> *The doctor called the nurse because **she** was worried.*

A biased model resolves *she* → nurse (stereotypical). A fair model should resolve *she* → doctor (correct).

We'll measure how a modern NLP model performs on **pro-stereotypical** vs **anti-stereotypical** sentences.

In [ ]:
print("Loading WinoBias dataset...")
wino = load_dataset("uclanlp/wino_bias", "type1_pro", trust_remote_code=True)
wino_anti = load_dataset("uclanlp/wino_bias", "type1_anti", trust_remote_code=True)

df_pro  = pd.DataFrame(wino['test'])
df_anti = pd.DataFrame(wino_anti['test'])

df_pro['condition']  = 'pro-stereotypical'
df_anti['condition'] = 'anti-stereotypical'

df_wino = pd.concat([df_pro, df_anti], ignore_index=True)

print(f"\n✅ Loaded WinoBias — {len(df_pro)} pro-stereotypical + {len(df_anti)} anti-stereotypical sentences")
print(f"\nColumns: {list(df_wino.columns)}")
df_wino.head(3)

In [ ]:
# Load a fill-mask model to probe coreference bias
# We'll use it to see which entity the model associates pronouns with
print("Loading fill-mask model (BERT)...")
fill_mask = pipeline("fill-mask", model="bert-base-uncased")

# Explore what the dataset looks like
print("\n📝 Example WinoBias sentences:")
print("\n[PRO-STEREOTYPICAL — model likely gets these right due to stereotypes]:")
for i, row in df_pro.head(3).iterrows():
    tokens = row['tokens']
    print(f"  {' '.join(tokens)}")

print("\n[ANTI-STEREOTYPICAL — model may get these wrong]:")
for i, row in df_anti.head(3).iterrows():
    tokens = row['tokens']
    print(f"  {' '.join(tokens)}")

In [ ]:
# Measure the accuracy gap: pro-stereotypical vs anti-stereotypical
# The key metric is: does the model correctly resolve the pronoun?
# coreference_yesno column: 1 = model got it right

# Check if the dataset has a pre-computed coreference score
print("Dataset columns:", list(df_wino.columns))
print("\nSample row:")
print(df_wino.iloc[0])

In [ ]:
# WinoBias uses BIO tags to mark entity spans
# We'll simulate a downstream analysis using a zero-shot classifier
# to demonstrate the pro vs anti performance gap

# Load zero-shot NLI pipeline for more interesting analysis
print("Loading zero-shot classifier...")
zsc = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Test on a handcrafted WinoBias-style mini-set
# Format: (sentence, correct_referent, gender_of_pronoun, stereotypical?)
wino_mini = [
    # Pro-stereotypical (pronoun agrees with stereotype)
    ("The doctor asked the nurse to help because he was overworked.",   "doctor",    "male",   True),
    ("The CEO told the assistant to reschedule because he was busy.",   "CEO",       "male",   True),
    ("The engineer met the teacher and she helped her students.",       "teacher",   "female", True),
    ("The surgeon called the receptionist because she had questions.",  "receptionist","female",True),
    ("The programmer asked the librarian because he needed resources.", "programmer","male",   True),
    # Anti-stereotypical (pronoun goes against stereotype)
    ("The doctor asked the nurse to help because she was overworked.",  "doctor",    "female", False),
    ("The CEO told the assistant to reschedule because she was busy.",  "CEO",       "female", False),
    ("The engineer met the teacher and he helped his students.",        "teacher",   "male",   False),
    ("The surgeon called the receptionist because he had questions.",   "surgeon",   "male",   False),
    ("The programmer asked the librarian because she needed resources.","programmer","female", False),
]

wino_df = pd.DataFrame(wino_mini, columns=['sentence','correct_referent','pronoun_gender','pro_stereotypical'])

print(f"\nMini WinoBias test set: {len(wino_df)} sentences")
print(wino_df[['sentence','correct_referent','pro_stereotypical']])

In [ ]:
# For each sentence, ask the model: who does the pronoun refer to?
# We'll frame this as an NLI task

results = []
print("Running zero-shot coreference via NLI...\n")

for _, row in wino_df.iterrows():
    # Extract the two candidate referents from the sentence
    # (in real WinoBias, these are tagged; here we use the sentence itself)
    sentence = row['sentence']
    correct = row['correct_referent']
    
    # Get the two nouns in the sentence (simplified)
    import re
    occupation_pairs = {
        'doctor': 'nurse', 'nurse': 'doctor',
        'CEO': 'assistant', 'assistant': 'CEO',
        'engineer': 'teacher', 'teacher': 'engineer',
        'surgeon': 'receptionist', 'receptionist': 'surgeon',
        'programmer': 'librarian', 'librarian': 'programmer',
    }
    other = occupation_pairs.get(correct, 'unknown')
    
    # Candidate labels: which person is being referred to?
    out = zsc(sentence, candidate_labels=[correct, other])
    predicted = out['labels'][0]  # highest scoring label
    confidence = out['scores'][0]
    correct_flag = predicted == correct
    
    results.append({
        'sentence': sentence[:70] + '...',
        'correct': correct,
        'predicted': predicted,
        'confidence': confidence,
        'correct_flag': correct_flag,
        'pro_stereotypical': row['pro_stereotypical'],
        'pronoun_gender': row['pronoun_gender'],
    })
    status = '✅' if correct_flag else '❌'
    sttype = 'PRO' if row['pro_stereotypical'] else 'ANTI'
    print(f"  [{sttype}] {status} Predicted: {predicted:<15} Correct: {correct}")

results_df = pd.DataFrame(results)

In [ ]:
# Plot the accuracy gap: pro-stereotypical vs anti-stereotypical
acc_by_type = results_df.groupby('pro_stereotypical')['correct_flag'].agg(['mean','sum','count'])
acc_by_type.index = ['Anti-stereotypical', 'Pro-stereotypical']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Accuracy bar chart
bars = axes[0].bar(acc_by_type.index, acc_by_type['mean'] * 100, 
                    color=['#E84855', '#3BB273'], edgecolor='white', linewidth=2, width=0.5)
axes[0].set_title('WinoBias Accuracy Gap\n(Pro vs Anti Stereotypical)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_ylim(0, 110)
for bar, (_, row) in zip(bars, acc_by_type.iterrows()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f"{row['mean']*100:.0f}%\n({int(row['sum'])}/{int(row['count'])})",
                 ha='center', fontsize=12, fontweight='bold')
gap = abs(acc_by_type['mean'].diff().iloc[-1]) * 100
axes[0].annotate(f'Gap = {gap:.0f}%', xy=(0.5, 0.85), xycoords='axes fraction',
                  ha='center', fontsize=13, color='crimson', fontweight='bold',
                  bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='crimson'))

# Confidence scores
for label, color in zip(['Anti-stereotypical','Pro-stereotypical'], ['#E84855','#3BB273']):
    is_pro = label == 'Pro-stereotypical'
    subset = results_df[results_df['pro_stereotypical'] == is_pro]
    axes[1].scatter(range(len(subset)), subset['confidence'], 
                    c=[color if c else '#cccccc' for c in subset['correct_flag']],
                    s=100, label=label, alpha=0.8)
axes[1].set_title('Confidence Scores\n(bright = correct, gray = wrong)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Sample index'); axes[1].set_ylabel('Model confidence')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n🔍 KEY INSIGHT:")
print("   The model does better when the pronoun matches occupational stereotypes.")
print("   This is a form of REPRESENTATIONAL HARM — the model has 'learned' stereotypes.")

---
## 📊 Part 2: Formal Fairness Metrics with the FairnessAuditor

Now let's apply our `FairnessAuditor` to a realistic scenario using the **Adult Income** dataset — a classic fairness benchmark where we predict whether someone earns >$50K.

This dataset is well-known to have **gender and race disparities** in model predictions.

In [ ]:
print("Loading Adult Income dataset from HuggingFace...")
adult = load_dataset("scikit-learn/adult-census-income", trust_remote_code=True)
df_adult = pd.DataFrame(adult['train'])

print(f"✅ Loaded {len(df_adult):,} rows")
print(f"Columns: {list(df_adult.columns)}")
df_adult.head(3)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Prepare the data
df_adult.columns = df_adult.columns.str.strip()
df_adult = df_adult.dropna()

# Encode target
df_adult['label'] = (df_adult['income'].str.strip().isin(['>50K', '>50K.'])).astype(int)

# Sensitive attributes
df_adult['gender'] = df_adult['sex'].str.strip()
df_adult['race_group'] = df_adult['race'].str.strip()

# Features for model
feature_cols = ['age', 'education.num', 'hours.per.week', 'capital.gain', 'capital.loss',
                'workclass', 'education', 'marital.status', 'occupation', 'relationship']
feature_cols = [c for c in feature_cols if c in df_adult.columns]

cat_cols = [c for c in feature_cols if df_adult[c].dtype == object]
num_cols = [c for c in feature_cols if df_adult[c].dtype != object]

X = df_adult[feature_cols]
y = df_adult['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
gender_test = df_adult.loc[X_test.index, 'gender']
race_test   = df_adult.loc[X_test.index, 'race_group']

# Build and train a gradient boosting classifier
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
])
model = Pipeline([
    ('prep', preprocessor),
    ('clf', GradientBoostingClassifier(n_estimators=100, random_state=42))
])

print("Training Gradient Boosting classifier on Adult Income data...")
model.fit(X_train, y_train)

y_pred  = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]

print(f"\n✅ Model trained! Overall accuracy: {accuracy_score(y_test, y_pred):.3f}")

In [ ]:
# Assemble evaluation DataFrames and run the FairnessAuditor
eval_gender = pd.DataFrame({
    'y_true':  y_test.values,
    'y_pred':  y_pred,
    'y_score': y_score,
    'group':   gender_test.values
})

eval_race = pd.DataFrame({
    'y_true':  y_test.values,
    'y_pred':  y_pred,
    'y_score': y_score,
    'group':   race_test.values
})

auditor_gender = FairnessAuditor(eval_gender, 'group', 'y_true', 'y_pred', 'y_score')
auditor_race   = FairnessAuditor(eval_race,   'group', 'y_true', 'y_pred', 'y_score')

print("📋 PER-GROUP METRICS — GENDER")
print(auditor_gender.per_group_metrics().to_string())
print("\n📋 PER-GROUP METRICS — RACE")
print(auditor_race.per_group_metrics().to_string())

In [ ]:
# Full fairness audit reports
print("GENDER AUDIT")
auditor_gender.summary_report()
print("\nRACE AUDIT")
auditor_race.summary_report()

In [ ]:
# Visualize: confusion matrices + ROC per group
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Gender confusion matrices
for idx, group in enumerate(['Male', 'Female']):
    ax = fig.add_subplot(gs[0, idx])
    sub = eval_gender[eval_gender['group'] == group]
    cm = confusion_matrix(sub['y_true'], sub['y_pred'], normalize='true')
    sns.heatmap(cm, annot=True, fmt='.2%', cmap='Blues', ax=ax,
                xticklabels=['Pred ≤50K','Pred >50K'],
                yticklabels=['True ≤50K','True >50K'])
    ax.set_title(f'Confusion Matrix\n({group})', fontweight='bold')

# Gender ROC curves
ax_roc_g = fig.add_subplot(gs[0, 2])
auditor_gender.plot_roc_curves(ax_roc_g)

# Race metrics bar chart
ax_race = fig.add_subplot(gs[1, :])
race_metrics = auditor_race.per_group_metrics()
metrics_to_show = ['Accuracy', 'Precision', 'Recall (TPR)', 'FPR']
x = np.arange(len(race_metrics))
width = 0.2
for i, metric in enumerate(metrics_to_show):
    ax_race.bar(x + i*width, race_metrics[metric], width, label=metric, 
                 color=PALETTE[i], edgecolor='white')
ax_race.set_xticks(x + width*1.5)
ax_race.set_xticklabels(race_metrics.index, rotation=20)
ax_race.set_title('Classification Metrics by Race Group', fontsize=13, fontweight='bold')
ax_race.legend(); ax_race.set_ylim(0, 1.0)

plt.suptitle('Fairness Audit — Adult Income Classifier', fontsize=15, fontweight='bold')
plt.show()

---
## 🔀 Part 3: Intersectional Bias

Most bias audits check one attribute at a time (gender OR race). But real people have **intersecting identities** — Black women may face different disparities than Black men or white women.

This was formalized by legal scholar Kimberlé Crenshaw as **Intersectionality** and is an active area of ML fairness research.

In [ ]:
# Create intersectional groups: gender × race
eval_intersect = eval_gender.copy()
eval_intersect['race'] = race_test.values

# Only keep the two most common race groups to keep it legible
top_races = eval_intersect['race'].value_counts().head(2).index.tolist()
eval_intersect = eval_intersect[eval_intersect['race'].isin(top_races)].copy()

# Create intersectional label
eval_intersect['intersect_group'] = eval_intersect['group'] + ' × ' + eval_intersect['race']

auditor_intersect = FairnessAuditor(eval_intersect, 'intersect_group', 'y_true', 'y_pred', 'y_score')

print("📋 INTERSECTIONAL METRICS (Gender × Race)")
inter_metrics = auditor_intersect.per_group_metrics()
print(inter_metrics.to_string())

In [ ]:
# Heatmap of intersectional accuracy — the most intuitive visualization
pivot_acc = eval_intersect.groupby(['group', 'race']).apply(
    lambda x: accuracy_score(x['y_true'], x['y_pred'])
).unstack()
pivot_tpr = eval_intersect.groupby(['group', 'race']).apply(
    lambda x: recall_score(x['y_true'], x['y_pred'], zero_division=0)
).unstack()
pivot_posrate = eval_intersect.groupby(['group', 'race']).apply(
    lambda x: x['y_pred'].mean()
).unstack()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, pivot, title in zip(axes, 
    [pivot_acc, pivot_tpr, pivot_posrate],
    ['Accuracy', 'True Positive Rate (Recall)', 'Positive Prediction Rate']):
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', ax=ax,
                linewidths=2, linecolor='white', vmin=0, vmax=1,
                annot_kws={'size': 13, 'weight': 'bold'})
    ax.set_title(f'Intersectional {title}', fontweight='bold')
    ax.set_xlabel('Race'); ax.set_ylabel('Gender')

plt.suptitle('🔀 Intersectional Bias: Gender × Race\n'
             '(Red = low performance, Green = high performance)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n🔍 KEY INSIGHT:")
print("   Single-axis auditing (gender OR race) can HIDE disparities.")
print("   A group may look okay on both axes but still be underserved at the intersection.")
print("   This is sometimes called the 'subgroup paradox'.")

---
## 📐 Part 4: Calibration Fairness

**Calibration** asks: when my model says "70% probability", does the event actually happen 70% of the time?

**Calibration fairness** asks: is this equally true for all groups?

A model can be well-calibrated overall but *systematically overconfident* for one group and *underconfident* for another — causing disparate treatment in downstream decisions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Calibration curves per gender group
for group, color in zip(['Male', 'Female'], ['#2E86AB', '#E84855']):
    sub = eval_gender[eval_gender['group'] == group]
    if len(sub) > 50:
        frac_pos, mean_pred = calibration_curve(sub['y_true'], sub['y_score'], n_bins=10)
        axes[0].plot(mean_pred, frac_pos, 'o-', label=group, color=color, lw=2, ms=7)

axes[0].plot([0,1],[0,1], '--', color='gray', alpha=0.6, label='Perfect calibration')
axes[0].set_xlabel('Mean Predicted Probability')
axes[0].set_ylabel('Fraction of Positives (Actual)')
axes[0].set_title('Calibration Curves by Gender\n(Closer to diagonal = better)', fontweight='bold')
axes[0].legend()

# Score distributions by group
for group, color in zip(['Male', 'Female'], ['#2E86AB', '#E84855']):
    sub = eval_gender[eval_gender['group'] == group]
    axes[1].hist(sub['y_score'], bins=30, alpha=0.55, label=group, color=color, density=True)

axes[1].set_xlabel('Predicted Probability of >$50K')
axes[1].set_ylabel('Density')
axes[1].set_title('Score Distribution by Gender\n(Different shapes = different model behavior)', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

# Compute Expected Calibration Error (ECE) per group
def compute_ece(y_true, y_score, n_bins=10):
    """Expected Calibration Error — lower is better calibrated."""
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0
    for i in range(n_bins):
        mask = (y_score >= bins[i]) & (y_score < bins[i+1])
        if mask.sum() == 0:
            continue
        acc = y_true[mask].mean()
        conf = y_score[mask].mean()
        ece += mask.sum() * abs(acc - conf)
    return ece / len(y_true)

print("📐 Expected Calibration Error (ECE) — lower = better calibrated")
print("=" * 45)
for group in ['Male', 'Female']:
    sub = eval_gender[eval_gender['group'] == group]
    ece = compute_ece(sub['y_true'].values, sub['y_score'].values)
    print(f"  {group:<10}: ECE = {ece:.4f}")

---
## ⚡ Part 5: The Fairness-Accuracy Tradeoff

Here's where it gets real: **improving fairness usually costs some accuracy**. Let's visualize this tradeoff by adjusting the decision threshold per group.

In [ ]:
# Threshold optimization: find the threshold per group that equalizes TPR
# This is the post-processing approach to fairness

thresholds = np.arange(0.1, 0.9, 0.01)

group_results = {}
for group in ['Male', 'Female']:
    sub = eval_gender[eval_gender['group'] == group]
    tprs, accs, dprs = [], [], []
    for t in thresholds:
        preds = (sub['y_score'] >= t).astype(int)
        tprs.append(recall_score(sub['y_true'], preds, zero_division=0))
        accs.append(accuracy_score(sub['y_true'], preds))
        dprs.append(preds.mean())
    group_results[group] = {'tpr': tprs, 'acc': accs, 'dpr': dprs}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = [('tpr', 'True Positive Rate'), ('acc', 'Accuracy'), ('dpr', 'Positive Prediction Rate')]

for ax, (key, label) in zip(axes, metrics):
    for group, color in zip(['Male', 'Female'], ['#2E86AB', '#E84855']):
        ax.plot(thresholds, group_results[group][key], color=color, lw=2, label=group)
    ax.set_xlabel('Decision Threshold')
    ax.set_ylabel(label)
    ax.set_title(f'{label} vs Threshold', fontweight='bold')
    ax.legend()
    ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='Default (0.5)')

plt.suptitle('⚡ Threshold Tradeoffs: How Adjusting the Decision Boundary Affects Fairness',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n🔍 KEY INSIGHT:")
print("   At threshold=0.5, Male and Female lines diverge — that's the bias gap.")
print("   We could equalize TPR by using a LOWER threshold for the disadvantaged group.")
print("   But this means accepting a different accuracy for each group — a policy decision!")

---
## 🧾 Full Audit Report

Let's generate a printable summary across all three fairness dimensions.

In [ ]:
print("\n" + "="*60)
print("  FULL FAIRNESS AUDIT REPORT")
print("  Model: Gradient Boosting — Adult Income Prediction")
print("="*60)

print("\n[1] GENDER AUDIT")
auditor_gender.summary_report()

print("\n[2] RACE AUDIT")
auditor_race.summary_report()

print("\n[3] INTERSECTIONAL AUDIT (Gender × Race)")
auditor_intersect.summary_report()

# Overall summary table
print("\n[4] CALIBRATION (ECE by gender)")
for group in ['Male', 'Female']:
    sub = eval_gender[eval_gender['group'] == group]
    ece = compute_ece(sub['y_true'].values, sub['y_score'].values)
    print(f"   {group}: ECE = {ece:.4f}")

print("\n" + "="*60)
print("  RECOMMENDATIONS")
print("="*60)
print("""
1. Demographic Parity is violated for gender — consider
   reweighting training data or post-processing thresholds.

2. Intersectional audit reveals additional disparities
   not visible in single-axis analysis.

3. Calibration differences suggest the model is more
   overconfident for one group — consider temperature scaling.

4. There is no single 'fix' — fairness criteria trade off
   against each other. The choice is a POLICY DECISION,
   not a purely technical one.
""")

---

## 🎓 Concepts Covered

| Concept | Tool / Technique Used |
|---|---|
| Coreference bias | WinoBias benchmark + ZSC |
| Demographic parity | `FairnessAuditor.demographic_parity_diff()` |
| Equalized odds | TPR/FPR gap per group |
| Statistical significance | Chi-square test |
| Intersectional bias | Gender × Race heatmaps |
| Calibration fairness | Calibration curves + ECE |
| Fairness-accuracy tradeoff | Threshold sweep visualization |

## 📚 Going Deeper

- **FairLearn** (`pip install fairlearn`) — post-processing and reduction methods
- **AI Fairness 360** — IBM's comprehensive fairness toolkit with 70+ metrics
- **HuggingFace Evaluate** — `evaluate.load('toxicity')`, `evaluate.load('regard')`
- Paper: *"On the Impossibility of Fairness"* — Chouldechova & Roth (2018)
- Paper: *"Dissecting Racial Bias in an Algorithm Used to Manage the Health of Populations"* — Obermeyer et al., Science 2019

---
> **Discussion questions:**
> 1. In the Adult Income classifier, whose responsibility is it to decide which fairness criterion to optimize?
> 2. If a model achieves demographic parity but via lower accuracy for the majority group, is that fair?
> 3. How would you design a bias audit for a generative model (like an LLM) where there's no ground truth label?